# Toffoli Gate (CCX)

The **Toffoli gate** is a 3-qubit gate that flips the target qubit only when **both** control qubits are $|1\rangle$. It is universal for classical computation and can implement quantum AND.

In [ ]:
import pennylane as qml
import numpy as np

## Define circuits for individual cases

In [ ]:
dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def toffoli_000():
    qml.Toffoli(wires=[0, 1, 2])
    return qml.state()

@qml.qnode(dev)
def toffoli_110():
    qml.PauliX(wires=0)
    qml.PauliX(wires=1)
    qml.Toffoli(wires=[0, 1, 2])
    return qml.state()

@qml.qnode(dev)
def toffoli_100():
    qml.PauliX(wires=0)
    qml.Toffoli(wires=[0, 1, 2])
    return qml.state()

## Individual cases

In [ ]:
cases = [
    ("|000> -> |000>", toffoli_000),
    ("|110> -> |111> (both controls=1, target flips)", toffoli_110),
    ("|100> -> |100> (only one control=1, no flip)", toffoli_100),
]

for label, fn in cases:
    sv = fn()
    probs = np.abs(sv) ** 2
    outcome = np.argmax(probs)
    bits = format(outcome, "03b")
    print(f"{label}")
    print(f"  dominant: |{bits}> (p = {probs[outcome]:.4f})")
    print()

## Full truth table

In [ ]:
@qml.qnode(dev)
def toffoli_all_inputs():
    return qml.probs(wires=[0, 1, 2])

sv = toffoli_all_inputs()
probs = np.abs(sv) ** 2

print(f"  {'Input':>8s}  {'Output':>8s}  {'P(output)':>10s}")
print(f"  {'-'*8}  {'-'*8}  {'-'*10}")
for i in range(8):
    inp = format(i, "03b")
    c0 = (i >> 2) & 1
    c1 = (i >> 1) & 1
    t = i & 1
    out_t = t ^ (c0 & c1)
    out = (c0 << 2) | (c1 << 1) | out_t
    out_bits = format(out, "03b")
    print(f"  |{inp}>  ->  |{out_bits}>  {probs[i]:.4f}")

## Toffoli as a quantum AND gate

Initialize q2 in $|1\rangle$ (ancilla), then Toffoli computes $\text{AND}(a, b)$ into q2.

In [ ]:
dev_and = qml.device("default.qubit", wires=3)

@qml.qnode(dev_and)
def quantum_and(a, b):
    if a:
        qml.PauliX(wires=0)
    if b:
        qml.PauliX(wires=1)
    qml.PauliX(wires=2)
    qml.Toffoli(wires=[0, 1, 2])
    return qml.probs(wires=2)

print("Toffoli as AND gate (q2 initialized to |1>):")
for a in [0, 1]:
    for b in [0, 1]:
        p = quantum_and(a, b)
        print(f"  AND({a}, {b}) = {np.argmax(p)}")